# Part 3 - Adversarial Attacks

Implements two attacks from scratch:
1. Character-level evasion attack
2. Label-flipping poisoning attack

In [1]:
import json
import numpy as np
import pandas as pd

from transformers import AutoModelForSequenceClassification, AutoTokenizer

from src.attack_utils import perturb_many, compute_attack_success_rate, poison_flip_labels
from src.config import BASE_MODEL_NAME, DATA_PATH, MAX_LENGTH, MODELS_DIR, RANDOM_STATE, SPLIT_PATH
from src.data_utils import load_dataset, load_split_indices, build_subsets_from_indices
from src.metrics_utils import evaluate_binary_classification
from src.model_utils import (
    predict_probabilities_from_model,
    train_distilbert,
)

In [2]:
# Load split + baseline model
df = load_dataset(DATA_PATH)
split_payload = load_split_indices(SPLIT_PATH)
train_df, eval_df = build_subsets_from_indices(df, split_payload)

part1_model_dir = MODELS_DIR / 'part1_checkpoint'
tokenizer = AutoTokenizer.from_pretrained(part1_model_dir)
baseline_model = AutoModelForSequenceClassification.from_pretrained(part1_model_dir)

eval_probs = predict_probabilities_from_model(
    model=baseline_model,
    tokenizer=tokenizer,
    texts=eval_df['comment_text'].tolist(),
    max_length=MAX_LENGTH,
)
eval_df = eval_df.copy()
eval_df['clean_prob'] = eval_probs
eval_df['clean_pred'] = (eval_df['clean_prob'] >= 0.5).astype(int)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

## Attack 1 - Character-level Evasion

In [3]:
# Select 500 comments that baseline classifies as toxic with confidence >= 0.7
attack_pool = eval_df[(eval_df['clean_pred'] == 1) & (eval_df['clean_prob'] >= 0.7)].copy()
sample_n = min(500, len(attack_pool))
attack_sample = attack_pool.sample(n=sample_n, random_state=RANDOM_STATE).reset_index(drop=True)
print('Selected attack sample size:', len(attack_sample))

perturbed_texts = perturb_many(attack_sample['comment_text'].tolist(), seed=RANDOM_STATE)
attack_sample['perturbed_text'] = perturbed_texts

Selected attack sample size: 500


In [4]:
perturbed_probs = predict_probabilities_from_model(
    model=baseline_model,
    tokenizer=tokenizer,
    texts=attack_sample['perturbed_text'].tolist(),
    max_length=MAX_LENGTH,
)

asr = compute_attack_success_rate(
    original_probs=attack_sample['clean_prob'].to_numpy(),
    attacked_probs=perturbed_probs,
    decision_threshold=0.5,
)

attack_table = pd.DataFrame({
    'metric': ['sample_size', 'attack_success_rate', 'avg_conf_before', 'avg_conf_after'],
    'value': [
        len(attack_sample),
        asr,
        attack_sample['clean_prob'].mean(),
        float(np.mean(perturbed_probs)),
    ]
})
display(attack_table)

attack_sample[['comment_text', 'perturbed_text']].head(5)

,metric,value
0,sample_size,500.000000
1,attack_success_rate,0.910000
2,avg_conf_before,0.963795
3,avg_conf_after,0.097983


,comment_text,perturbed_text
0,I’ll decide what’s best for me. A bunch of mor...,I’lll dеcidee whhааtt’s bbest fоrr mme. A bbuu...
1,I can only imagine how angry and mad they are ...,I caan onnlyy imagіnne hhоw anngryy аnd mad tt...
2,Asian countries for Asians.\n\nBlack countries...,Asіann ccоuntrriies foor Asіans.\n\nBlackk coo...
3,You should do your homework before making prof...,Уou shoulld doo youur homеwork bbefforee makin...
4,'\n\nThe buffoon Trump promised to build a wal...,'\n\nThe bbuffооn Trumр pprrоmіsedd too buiild...


## Attack 2 - Label-Flipping Poisoning

In [5]:
# Create poisoned training set: flip 5% labels
poisoned_train_df = poison_flip_labels(train_df, flip_fraction=0.05, seed=RANDOM_STATE)
print('Poisoned rows:', int(poisoned_train_df['is_poisoned_row'].sum()))

poison_model_dir = MODELS_DIR / 'part3_poisoned_checkpoint'
poison_model_dir.mkdir(parents=True, exist_ok=True)

# Retrain fresh model from original pretrained checkpoint with same hyperparameters as Part 1
poison_trainer, poison_tokenizer = train_distilbert(
    model_name=BASE_MODEL_NAME,
    train_df=poisoned_train_df,
    eval_df=eval_df,
    output_dir=poison_model_dir,
    max_length=MAX_LENGTH,
    num_train_epochs=3,
    learning_rate=2e-5,
    train_batch_size=16,
    eval_batch_size=32,
    seed=RANDOM_STATE,
)

Poisoned rows: 5000


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

d:\FAST\Semester 8\Responsible AI\Assignments\A2\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [6]:
# Evaluate poisoned model vs baseline on same clean eval set
poison_probs = predict_probabilities_from_model(
    model=poison_trainer.model,
    tokenizer=poison_tokenizer,
    texts=eval_df['comment_text'].tolist(),
    max_length=MAX_LENGTH,
)

baseline_metrics = evaluate_binary_classification(eval_df['label'].to_numpy(), eval_df['clean_prob'].to_numpy(), threshold=0.5)
poison_metrics = evaluate_binary_classification(eval_df['label'].to_numpy(), poison_probs, threshold=0.5)

def fnr(cm):
    tn, fp, fn, tp = cm.ravel()
    return fn / (fn + tp) if (fn + tp) else 0.0

comparison = pd.DataFrame([
    {
        'model': 'baseline',
        'accuracy': baseline_metrics['accuracy'],
        'f1_macro': baseline_metrics['f1_macro'],
        'fnr': fnr(baseline_metrics['confusion_matrix']),
    },
    {
        'model': 'poisoned',
        'accuracy': poison_metrics['accuracy'],
        'f1_macro': poison_metrics['f1_macro'],
        'fnr': fnr(poison_metrics['confusion_matrix']),
    },
])
display(comparison)

,model,accuracy,f1_macro,fnr
0,baseline,0.94455,0.810187,0.354597
1,poisoned,0.94820,0.808027,0.414009


## Required Interpretation

**Note:** The assignment refers to the label as `toxic`, but our dataset/code maps this to the column `target`. Toxicity is what we are detecting.

Compare operational risk of both attacks:
1. **Which attack is more dangerous?** The **Poisoning Attack** is generally more dangerous at an operational level. While evasion targets specific inputs at inference time, poisoning compromises the model itself during training. Poisoned models implicitly learn wrong associations (e.g., that a specific benign keyword is toxic), harming thousands of future decisions implicitly and invisibly. 
2. **Why does FNR jump for the poisoned model?** The poisoned dataset injected false correlations (e.g., incorrectly flipping labels or washing out features of real toxic text). Consequently, the model became more hesitant to flag actual toxic comments containing poisoned triggers as toxic, causing the False Negative Rate (FNR) to jump significantly and the overall recall of toxic content to drop.
\n